In [20]:
import numpy as np
import pandas as pd
from samba.dcerpc.dcerpc import RTS_FLAG_ECHO
from sklearn.model_selection import train_test_split
from feature_eng import preprocess
df = pd.read_csv("~/ML/Data/house_prices/train.csv")

x_train,x_test = train_test_split(df,test_size=0.2,random_state=42)

In [30]:
x_train,x_test,y_train,y_test = preprocess(x_train,x_test)

In [3]:
x_train.isna().mean()
x_test.isna().mean()

LotFrontage              0.0
LotArea                  0.0
Street                   0.0
LotShape                 0.0
Utilities                0.0
                        ... 
SaleCondition_AdjLand    0.0
SaleCondition_Alloca     0.0
SaleCondition_Family     0.0
SaleCondition_Normal     0.0
SaleCondition_Partial    0.0
Length: 189, dtype: float64

## Training Random Forrest without any filtering

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV,KFold,cross_val_score
from sklearn.pipeline import Pipeline

kf = KFold(n_splits=5,random_state=42,shuffle=True)

param_grid = {
    'n_estimators': range(100,400,100),
    'max_depth': range(1,10,1),
    'min_samples_split': range(2,20,2),
    'min_samples_leaf': range(1,10,1),
    'criterion': ['squared_error'],
}

grid_search = RandomizedSearchCV(
    RandomForestRegressor(),
    param_distributions=param_grid,   # note: param_distributions, not param_grid
    n_iter=50,                        # goes here
    cv=kf,
    scoring='neg_root_mean_squared_error',
    verbose=2,
    return_train_score=True,
    random_state=42
)


In [5]:
result = grid_search.fit(x_train,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   2.1s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   2.0s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.8s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.8s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.8s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   2.5s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   2.4s
[CV] END criterion=squared_error, max_depth=6, min_samples_le

## Training with Correlation Filter

In [10]:
from feature_eng import correlation_filter

x_train_filtered = correlation_filter(x_train,y_train,0.7)

In [11]:
result_filtered = grid_search.fit(x_train_filtered,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.2s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.2s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.2s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.3s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.2s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   1.6s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   1.6s
[CV] END criterion=squared_error, max_depth=6, min_samples_le

## Logging both of those

In [14]:
import dagshub
import mlflow
from sklearn.metrics import root_mean_squared_error
dagshub.init(repo_owner='ldavi22', repo_name='cs229-231', mlflow=True)

mlflow.set_tracking_uri("https://dagshub.com/ldavi22/cs229-231.mlflow")
mlflow.set_experiment('house-prices-linear-regression_')

with mlflow.start_run(run_name="Random Forrest Regressor with Correlation filter"):

    x_test = x_test[x_train_filtered.columns]
    best_idx = result_filtered.best_index_
    train_rmse = -result_filtered.cv_results_['mean_train_score'][best_idx]
    val_rmse = -result_filtered.best_score_
    val_std = result_filtered.cv_results_['std_test_score'][best_idx]
    test_pred = result_filtered.best_estimator_.predict(x_test)
    test_rmse = root_mean_squared_error(y_test,test_pred)

    mlflow.log_param("model", "RandomForrestRegressor")
    mlflow.log_param("n_features", x_train.shape[1])
    mlflow.log_param("n_samples", x_train.shape[0])
    mlflow.log_param("feature_selection", "Correlation filter")

    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_std", val_std)
    mlflow.log_metric("overfit_gap", val_rmse - train_rmse)

    for key,value in result.best_params_.items():
        mlflow.log_param(key, value)


Initialized MLflow to track repo "ldavi22/cs229-231"

Repository ldavi22/cs229-231 initialized!

🏃 View run Random Forrest Regressor with Correlation filter at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1/runs/1d86fc52723e489280ae6facebfdca28
🧪 View experiment at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1


In [8]:
import dagshub
import mlflow
from sklearn.metrics import root_mean_squared_error
dagshub.init(repo_owner='ldavi22', repo_name='cs229-231', mlflow=True)

mlflow.set_tracking_uri("https://dagshub.com/ldavi22/cs229-231.mlflow")
mlflow.set_experiment('house-prices-linear-regression_')

with mlflow.start_run(run_name="Random Forrest Regressor"):

    best_idx = result.best_index_
    train_rmse = -result.cv_results_['mean_train_score'][best_idx]
    val_rmse = -result.best_score_
    val_std = result.cv_results_['std_test_score'][best_idx]
    test_pred = result.best_estimator_.predict(x_test)
    test_rmse = root_mean_squared_error(y_test,test_pred)

    mlflow.log_param("model", "RandomForrestRegressor")
    mlflow.log_param("n_features", x_train.shape[1])
    mlflow.log_param("n_samples", x_train.shape[0])
    mlflow.log_param("feature_selection", "RandomForestRegressor")

    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_std", val_std)
    mlflow.log_metric("overfit_gap", val_rmse - train_rmse)

    for key,value in result.best_params_.items():
        mlflow.log_param(key, value)


Initialized MLflow to track repo "ldavi22/cs229-231"

Repository ldavi22/cs229-231 initialized!

🏃 View run Random Forrest Regressor at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1/runs/6876f055dfeb4defb79989f4eb12cd1a
🧪 View experiment at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1


## Filtering with SHAP Values

In [36]:
from logging_helper import log

result = grid_search.fit(x_train,y_train)


Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.4s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.4s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.3s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.3s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.3s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   1.7s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   1.8s
[CV] END criterion=squared_error, max_depth=6, min_samples_le

In [7]:
import shap

best_rf = grid_search.best_estimator_

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(x_test)

In [9]:
shap_importance = pd.DataFrame({'feature' : x_train.columns, 'importance' : np.abs(shap_values).mean(axis=0)}).sort_values('importance', ascending=False)

important_features = shap_importance.head(50)['feature'].tolist()
x_train_shap = x_train[important_features]

In [10]:
result_shap = grid_search.fit(x_train_shap,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.4s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.3s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.3s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.4s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.5s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   1.8s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   1.8s
[CV] END criterion=squared_error, max_depth=6, min_samples_le

In [12]:
from logging_helper import log

log(x_train_shap,y_train,x_test[x_train_shap.columns],y_test,result_shap,"Random Forrest with SHAP filtering")

Accessing as ldavi22

Initialized MLflow to track repo "ldavi22/cs229-231"

Repository ldavi22/cs229-231 initialized!

🏃 View run Random Forrest with SHAP filtering at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1/runs/708a9b31a78f45edb66f598301e6260a
🧪 View experiment at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1


## Using SHAP values for SHAP filtering

In [15]:
import shap

best_rf = grid_search.best_estimator_

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(x_test)

In [29]:
import pandas as pd
import numpy as np

# Mean absolute SHAP value per feature = how much it contributes on average
shap_importance = pd.DataFrame({
    'feature': x_train_filtered.columns,
    'importance': np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False)

print(shap_importance.head(20))
print(f'\nFeatures with importance > 0.001: {(shap_importance["importance"] > 0.001).sum()}')
print(f'Features with importance < 0.001: {(shap_importance["importance"] < 0.001).sum()}')

               feature  importance
6          OverallQual    0.196349
25           GrLivArea    0.077936
19         TotalBsmtSF    0.036930
36          GarageCars    0.031294
16          BsmtFinSF1    0.015116
8            YearBuilt    0.011816
1              LotArea    0.011139
34         FireplaceQu    0.010620
9         YearRemodAdd    0.009692
35        GarageFinish    0.009425
21          CentralAir    0.008611
7          OverallCond    0.006976
15        BsmtFinType1    0.005576
143  GarageType_Attchd    0.005386
23            2ndFlrSF    0.005087
12            BsmtQual    0.005027
40         OpenPorchSF    0.003324
37          GarageQual    0.002927
59         MSZoning_RM    0.002877
18           BsmtUnfSF    0.002825

Features with importance > 0.001: 31
Features with importance < 0.001: 130


In [26]:
top_features = shap_importance.head(50)['feature'].tolist()
x_train_shap = x_train_filtered[top_features]
x_test_shap = x_test[top_features]
x_train_shap.shape
y_test.shape

(1168, 50)

In [27]:
result_filtered = grid_search.fit(x_train_shap,y_train)


Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.1s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.0s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   0.9s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.0s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   0.9s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   1.2s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   1.2s
[CV] END criterion=squared_error, max_depth=6, min_samples_le

In [30]:
import dagshub
import mlflow
from sklearn.metrics import root_mean_squared_error
dagshub.init(repo_owner='ldavi22', repo_name='cs229-231', mlflow=True)

mlflow.set_tracking_uri("https://dagshub.com/ldavi22/cs229-231.mlflow")
mlflow.set_experiment('house-prices-linear-regression_')

with mlflow.start_run(run_name="Random Forrest Regressor with Correlation filter + SHAP"):

    best_idx = result_filtered.best_index_
    train_rmse = -result_filtered.cv_results_['mean_train_score'][best_idx]
    val_rmse = -result_filtered.best_score_
    val_std = result_filtered.cv_results_['std_test_score'][best_idx]
    test_pred = result_filtered.best_estimator_.predict(x_test_shap)
    test_rmse = root_mean_squared_error(y_test,test_pred)

    mlflow.log_param("model", "RandomForrestRegressor")
    mlflow.log_param("n_features", x_train.shape[1])
    mlflow.log_param("n_samples", x_train.shape[0])
    mlflow.log_param("feature_selection", "Correlation filter + SHAP")

    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_std", val_std)
    mlflow.log_metric("overfit_gap", val_rmse - train_rmse)

    for key,value in result.best_params_.items():
        mlflow.log_param(key, value)

Initialized MLflow to track repo "ldavi22/cs229-231"

Repository ldavi22/cs229-231 initialized!

🏃 View run Random Forrest Regressor with Correlation filter + SHAP at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1/runs/6fb5b7b3adf646af9e45b09cb593b4f7
🧪 View experiment at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1


## Feature Selection with RFE


In [13]:
from sklearn.feature_selection import RFECV

rf = RandomForestRegressor(n_estimators=100,random_state=42)

rfecv = RFECV(
    estimator=rf,
    step=1,
    cv=KFold(5, shuffle=True),
    scoring='neg_root_mean_squared_error',
    min_features_to_select=30,
    n_jobs=-1,
)

rfecv.fit(x_train,y_train)
print(f"Optimal features: {rfecv.n_features_}")


Optimal features: 55


In [14]:
x_train_rfe = rfecv.transform(x_train)

In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV,KFold,cross_val_score
from sklearn.pipeline import Pipeline

kf = KFold(n_splits=5,random_state=42,shuffle=True)

param_grid = {
    'n_estimators': range(100,400,100),
    'max_depth': range(1,10,1),
    'min_samples_split': range(2,20,2),
    'min_samples_leaf': range(1,10,1),
    'criterion': ['squared_error'],
}

grid_search = RandomizedSearchCV(
    RandomForestRegressor(),
    param_distributions=param_grid,   # note: param_distributions, not param_grid
    n_iter=50,                        # goes here
    cv=kf,
    scoring='neg_root_mean_squared_error',
    verbose=2,
    return_train_score=True,
    random_state=42
)


In [16]:
result = grid_search.fit(x_train_rfe,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.6s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.5s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.4s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.6s
[CV] END criterion=squared_error, max_depth=8, min_samples_leaf=6, min_samples_split=8, n_estimators=200; total time=   1.5s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   2.1s
[CV] END criterion=squared_error, max_depth=6, min_samples_leaf=4, min_samples_split=14, n_estimators=300; total time=   1.8s
[CV] END criterion=squared_error, max_depth=6, min_samples_le

In [31]:
selected_cols = x_train.columns[rfecv.support_]

x_train_rfe = pd.DataFrame(rfecv.transform(x_train),columns=selected_cols)
x_test_rfe = pd.DataFrame(rfecv.transform(x_test),columns=selected_cols)

In [32]:
log(x_train_rfe,y_train,x_test_rfe,y_test,result,"Random Forest with RFE")

Initialized MLflow to track repo "ldavi22/cs229-231"

Repository ldavi22/cs229-231 initialized!

/home/ldavi/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


🏃 View run Random Forest with RFE at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1/runs/f7cd0fea874f4f46bd53b1894ab493c6
🧪 View experiment at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1
